# 00 — Introducción NLP

In [1]:
from transformers import pipeline

# Montamos las tareas principales en el diccionario de clase
task_to_pipeline = {
    "sentiment-analysis": "sentiment-analysis",
    "ner": "ner",
    "summarization": "summarization"
}

# Probamos un modelo rápido en inglés para ver qué tal responde
pipeline_analisis = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

texto_test_1 = "This master program in business analytics is structured amazingly well!"
texto_test_2 = "The documentation is quite long, but I guess it covers what we need."

res_1 = pipeline_analisis(texto_test_1)
res_2 = pipeline_analisis(texto_test_2)

print("Test Positivo:", res_1)
print("Test Neutro/Ambiguo:", res_2)

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\Antonio\anaconda3\envs\UEA_IA\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Antonio\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Test Positivo: [{'label': 'POSITIVE', 'score': 0.9998635053634644}]
Test Neutro/Ambiguo: [{'label': 'NEGATIVE', 'score': 0.9224192500114441}]


# 01 — Embeddings básicos

In [2]:
import numpy as np
from gensim.models import Word2Vec

# --- Parte 1: One-Hot manual ---
mi_vocabulario = ["cat", "sat", "mat", "the"]
frase_tokens = ["the", "cat", "sat"]

# Mapeo rápido de palabras a posiciones
dicc_indices = {pal: pos for pos, pal in enumerate(mi_vocabulario)}
matriz_oh = np.zeros((len(frase_tokens), len(mi_vocabulario)))

for idx, pal in enumerate(frase_tokens):
    if pal in dicc_indices:
        matriz_oh[idx, dicc_indices[pal]] = 1

print("Nuestra matriz One-Hot:\n", matriz_oh)

# --- Parte 2: Similitud Coseno a mano ---
def calcular_coseno(v1, v2):
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    return np.dot(v1, v2) / (n1 * n2) if n1 > 0 and n2 > 0 else 0.0

# --- Parte 3: Word2Vec mini ---
textos_entrenamiento = [
    ["the", "cat", "sat", "on", "the", "mat"],
    ["the", "dog", "sat", "on", "the", "log"]
]
w2v_model = Word2Vec(textos_entrenamiento, vector_size=8, window=2, min_count=1)
print("Similitud cat-dog en W2V:", w2v_model.wv.similarity("cat", "dog"))

Nuestra matriz One-Hot:
 [[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 1. 0. 0.]]
Similitud cat-dog en W2V: -0.09075314


# 02 — Embeddings con Transformers

In [3]:
from transformers import AutoTokenizer, AutoModel
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

# Extraer hidden states con pooling manual
tokenizador = AutoTokenizer.from_pretrained("distilbert-base-uncased")
modelo_base = AutoModel.from_pretrained("distilbert-base-uncased")

inputs_ctx = tokenizador("Transformers build contextual embeddings.", return_tensors="pt")

with torch.no_grad():
    salidas = modelo_base(**inputs_ctx)

estados_ocultos = salidas.last_hidden_state
mascara_atencion = inputs_ctx['attention_mask'].unsqueeze(-1).expand(estados_ocultos.size()).float()

# Media de tokens sin contar el padding
media_embeddings = torch.sum(estados_ocultos * mascara_atencion, dim=1) / torch.clamp(mascara_atencion.sum(dim=1), min=1e-9)
print("Dimensiones del vector final:", media_embeddings.shape)

# Ahora con Sentence Transformers (mucho más limpio para producción)
s_trans = SentenceTransformer('all-MiniLM-L6-v2')
paraf_1 = "The stock market experienced a huge drop today."
paraf_2 = "Equities plummeted significantly during today's session."
frase_random = "My favorite recipe for chocolate cake requires three eggs."

vecs = s_trans.encode([paraf_1, paraf_2, frase_random])
sim_buena = np.dot(vecs[0], vecs[1]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1]))
sim_mala = np.dot(vecs[0], vecs[2]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[2]))

print(f"Similitud paráfrasis: {sim_buena:.4f} vs Tema distinto: {sim_mala:.4f}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dimensiones del vector final: torch.Size([1, 768])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Similitud paráfrasis: 0.4195 vs Tema distinto: -0.0458


# 03 — Ingeniería de prompts

In [4]:
# Definición del flujo estructurado en variables limpias
prompt_sistema = """Eres un asistente técnico de soporte corporativo. Tu objetivo es procesar la consulta del usuario y responder ÚNICAMENTE con un formato estructurado en JSON que contenga estas llaves:
{
  "answer": "Tu explicación concisa en español.",
  "confidence": un número entero del 1 al 10 según la certeza de los datos
}"""

prompt_usuario = "¿Cuáles son los requisitos de memoria para instalar Docker en Linux?"

# Template de Few-shot para clasificación
plantilla_reseñas = """Clasifica el tono de las opiniones del restaurante en: [positive, negative, neutral]

Opinión: El pescado estaba demasiado salado y el servicio fue lento.
Tono: negative

Opinión: Reservamos mesa para cuatro y nos sentaron rápido. El menú es el normal.
Tono: neutral

Opinión: {user_input}
Tono:"""

# 04 — Chatbots básicos

In [5]:
from typing import Any

# Prompt de rol socrático para la conversación
mensajes_iniciales = [
    {"role": "system", "content": "Actúa como un mentor de analítica de datos. No resuelvas los problemas directamente; haz preguntas guía para que el usuario deduzca la lógica."},
    {"role": "user", "content": "No entiendo por qué mi matriz de confusión da error al pasarle los datos."}
]

# Gestión del tamaño del historial para que no explote la API
def trim_history(messages: list[dict[str, Any]], max_turns: int) -> list[dict[str, Any]]:
    if not messages:
        return []
    
    # El prompt del sistema se queda sí o sí en el índice 0
    sys_msg = [m for m in messages if m.get("role") == "system"]
    mensajes_chat = [m for m in messages if m.get("role") != "system"]
    
    # Cada turno son 2 mensajes (user + assistant)
    limite_mensajes = max_turns * 2
    recorte = mensajes_chat[-limite_mensajes:] if limite_mensajes > 0 else []
    
    return sys_msg + recorte

# Prompt para forzar resúmenes intermedios
prompt_resumen = """Resume la conversación mantenida hasta ahora de forma súper compacta. Extrae solo los puntos técnicos bloqueantes y las soluciones intentadas.

Historial técnico:
{chat_history}

Resumen ejecutivo:"""

# 05 — Vectorstores y Retrieval

In [6]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# Chunker por longitud de caracteres
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 40) -> list[str]:
    lista_chunks = []
    salto = chunk_size - overlap
    for pos in range(0, len(text), salto):
        lista_chunks.append(text[pos:pos + chunk_size])
    return lista_chunks

# Indexación real en FAISS con Sentence Transformers
encoder_rag = SentenceTransformer('all-MiniLM-L6-v2')
textos_clase = [
    "El framework LangChain sirve para encadenar llamadas a LLMs.",
    "Las bases de datos vectoriales indexan embeddings para hacer búsquedas semánticas.",
    "El algoritmo de backpropagation se usa para entrenar redes neuronales artificiales."
]

vectores_docs = encoder_rag.encode(textos_clase)
faiss.normalize_L2(vectores_docs) # Para poder usar IndexFlatIP como coseno

idx_faiss = faiss.IndexFlatIP(vectores_docs.shape[1])
idx_faiss.add(vectores_docs)

# Consulta de prueba
pregunta_test = "Cómo busco información usando embeddings?"
vec_query = encoder_rag.encode([pregunta_test])
faiss.normalize_L2(vec_query)

scores, posiciones = idx_faiss.search(vec_query, k=2)
print("Documentos más cercanos recuperados:")
for p in posiciones[0]:
    print("->", textos_clase[p])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Documentos más cercanos recuperados:
-> Las bases de datos vectoriales indexan embeddings para hacer búsquedas semánticas.
-> El framework LangChain sirve para encadenar llamadas a LLMs.


# 06 — Introducción a RAG

In [7]:
def build_prompt(context_chunks: list[str], question: str) -> str:
    # Concatenamos los pasajes recuperados de la base de datos de vectores
    bloque_contexto = "\n".join([f"[Pasaje]: {c}" for c in context_chunks])
    
    prompt_completo = f"""Usa la información de los pasajes autorizados para responder la pregunta técnica del final. Si la respuesta no viene explícitamente en el contexto, di abiertamente 'No dispongo de datos suficientes'. No inventes.

### Context
{bloque_contexto}

### Question
{question}
"""
    return prompt_completo

# Simulación de test de RAG sin lanzar la llamada al LLM (Retrieval-Only)
chunks_recuperados = [
    "RAG es la abreviatura de Retrieval-Augmented Generation.",
    "Permite conectar datos externos a un modelo de lenguaje sin reentrenarlo."
]
print(build_prompt(chunks_recuperados, "¿Qué ventajas tiene usar RAG?"))

Usa la información de los pasajes autorizados para responder la pregunta técnica del final. Si la respuesta no viene explícitamente en el contexto, di abiertamente 'No dispongo de datos suficientes'. No inventes.

### Context
[Pasaje]: RAG es la abreviatura de Retrieval-Augmented Generation.
[Pasaje]: Permite conectar datos externos a un modelo de lenguaje sin reentrenarlo.

### Question
¿Qué ventajas tiene usar RAG?



### Gestión de fallos de cobertura en entornos reales

Si un usuario pregunta algo totalmente fuera de temario (por ejemplo, pedir una receta de cocina en un chatbot corporativo sobre Jira), podemos controlarlo antes de que el LLM alucine:

1. **Filtro por Umbral Crítico (Distance Thresholding):** Si al hacer el `idx_faiss.search()` la distancia/similitud del chunk más cercano es inferior a un score razonable (ej. menor a 0.45 en coseno), el backend cancela la operación y devuelve un mensaje estático: *"Lo siento, no encuentro información en los manuales oficiales sobre ese tema."*
2. **Instrucciones de Mitigación en el Prompt:** Forzar mediante diseño de prompts un comportamiento defensivo (como el prompt que programamos arriba con la cláusula de salvaguarda).

# 07 — Reranking y optimización

In [8]:
import numpy as np

# Datos simulados de la prueba técnica de latencias y scores
scores_bi_encoder = np.array([0.72, 0.81, 0.55, 0.78, 0.60])  
scores_cross_encoder = np.array([0.15, 0.92, 0.22, 0.88, 0.31])  

# 1. Recuperamos los 4 mejores según el Bi-Encoder (fase barata y rápida)
indices_top4_bi = np.argsort(scores_bi_encoder)[::-1][:4]

# 2. Reordenamos esos 4 candidatos usando las notas del Cross-Encoder (fase cara y precisa)
scores_rerank = scores_cross_encoder[indices_top4_bi]
ordenacion_local = np.argsort(scores_rerank)[::-1]
ranking_final = indices_top4_bi[ordenacion_local]

print("--- Ranking Final Optimizado (Two-Stage) ---")
for posicion, idx in enumerate(ranking_final):
    print(f"Puesto {posicion + 1}: Doc {idx} (Bi-Score: {scores_bi_encoder[idx]:.2f} | Cross-Score: {scores_cross_encoder[idx]:.2f})")

--- Ranking Final Optimizado (Two-Stage) ---
Puesto 1: Doc 1 (Bi-Score: 0.81 | Cross-Score: 0.92)
Puesto 2: Doc 3 (Bi-Score: 0.78 | Cross-Score: 0.88)
Puesto 3: Doc 4 (Bi-Score: 0.60 | Cross-Score: 0.31)
Puesto 4: Doc 0 (Bi-Score: 0.72 | Cross-Score: 0.15)
